# Problem Statement 10: Data Visualization III – Iris Dataset
**Objective:**
1. List features and their types.
2. Create histograms for each feature.
3. Create boxplots for each feature.
4. Compare distributions and identify outliers.

**Dataset Source:** https://archive.ics.uci.edu/ml/datasets/Iris

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris

sns.set_theme(style='whitegrid')
print("Libraries imported!")

## Step 1: Load the Iris Dataset

In [ ]:
iris_data = load_iris()
df = pd.DataFrame(iris_data.data, columns=iris_data.feature_names)
df['species'] = pd.Categorical.from_codes(iris_data.target, iris_data.target_names)

# Rename columns for readability
df.columns = ['Sepal_Length', 'Sepal_Width', 'Petal_Length', 'Petal_Width', 'Species']

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

## Step 2: List Features and Their Types

In [ ]:
feature_types = {
    'Sepal_Length': ('Numeric (continuous)', 'Length of the sepal in cm'),
    'Sepal_Width':  ('Numeric (continuous)', 'Width of the sepal in cm'),
    'Petal_Length': ('Numeric (continuous)', 'Length of the petal in cm'),
    'Petal_Width':  ('Numeric (continuous)', 'Width of the petal in cm'),
    'Species':      ('Nominal (categorical)', 'Species of the iris flower')
}

print(f"{'Feature':<15} {'Type':<30} {'Description'}")
print('-' * 75)
for feat, (ftype, desc) in feature_types.items():
    print(f"{feat:<15} {ftype:<30} {desc}")

print("\nData Types from pandas:")
print(df.dtypes)

In [ ]:
print("\nDataset Summary Statistics:")
df.describe()

In [ ]:
print("Species distribution:")
print(df['Species'].value_counts())

## Step 3: Histograms for Each Feature
Histograms show the frequency distribution of continuous variables.

In [ ]:
features = ['Sepal_Length', 'Sepal_Width', 'Petal_Length', 'Petal_Width']
colors   = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, (feat, color) in enumerate(zip(features, colors)):
    axes[i].hist(df[feat], bins=20, color=color, edgecolor='black', alpha=0.8)
    axes[i].axvline(df[feat].mean(), color='red', linestyle='--', lw=2, label=f"Mean: {df[feat].mean():.2f}")
    axes[i].axvline(df[feat].median(), color='blue', linestyle='-.', lw=2, label=f"Median: {df[feat].median():.2f}")
    axes[i].set_title(f'Histogram of {feat}')
    axes[i].set_xlabel(f'{feat} (cm)')
    axes[i].set_ylabel('Frequency')
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distributions – Iris Dataset', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Histograms by species (overlapping)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

species_colors = {'setosa': 'red', 'versicolor': 'green', 'virginica': 'blue'}

for i, feat in enumerate(features):
    for species, color in species_colors.items():
        subset = df[df['Species'] == species][feat]
        axes[i].hist(subset, bins=15, alpha=0.5, color=color, edgecolor='black', label=species)
    axes[i].set_title(f'{feat} by Species')
    axes[i].set_xlabel(f'{feat} (cm)')
    axes[i].set_ylabel('Frequency')
    axes[i].legend()

plt.suptitle('Feature Distributions by Species – Iris Dataset', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Step 4: Boxplots for Each Feature
Boxplots show the median, IQR, and outliers.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, feat in enumerate(features):
    sns.boxplot(x='Species', y=feat, data=df, palette='Set2', ax=axes[i])
    axes[i].set_title(f'Boxplot of {feat}')
    axes[i].set_xlabel('Species')
    axes[i].set_ylabel(f'{feat} (cm)')

plt.suptitle('Boxplots of Features by Species – Iris Dataset', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Overall boxplot without species split
plt.figure(figsize=(10, 5))
df[features].boxplot(patch_artist=True,
    boxprops=dict(facecolor='lightblue'),
    medianprops=dict(color='red', linewidth=2),
    whiskerprops=dict(linestyle='--'))
plt.title('Overall Feature Boxplots – Iris Dataset')
plt.ylabel('Value (cm)')
plt.tight_layout()
plt.show()

## Step 5: Outlier Detection

In [ ]:
print("Outlier Detection using IQR method:")
outlier_summary = []

for feat in features:
    Q1  = df[feat].quantile(0.25)
    Q3  = df[feat].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[feat] < lower) | (df[feat] > upper)]
    
    outlier_summary.append({
        'Feature': feat,
        'Q1': round(Q1, 3),
        'Q3': round(Q3, 3),
        'IQR': round(IQR, 3),
        'Lower Fence': round(lower, 3),
        'Upper Fence': round(upper, 3),
        'Num Outliers': len(outliers)
    })

outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df.to_string(index=False))

In [ ]:
# Correlation heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(df[features].corr(), annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5)
plt.title('Feature Correlation Heatmap – Iris Dataset')
plt.tight_layout()
plt.show()

## Summary of Observations

1. **Sepal Length:** Ranges 4.3 – 7.9 cm. Versicolor and Virginica overlap significantly. Setosa has the smallest sepal length.

2. **Sepal Width:** Setosa has the widest sepals. A few outliers observed in sepal width for versicolor.

3. **Petal Length and Width:** These are the most discriminative features. Setosa has very small petals; Virginica has the largest. Very clear separation between species.

4. **Outliers:** Sepal Width has the most outliers across species. Petal features have very few to no outliers.

5. **Correlation:** Petal Length and Petal Width are highly correlated (r ≈ 0.96). Sepal Width has negative correlation with the other features.

6. **Distribution Shape:** Sepal features are roughly normally distributed; petal features are bimodal due to the separation between Setosa and the other two species.